# Inspecting Dataset

By looking at the adata-object we find the following objects and information about the data:
_Shape_ : (48730, 29961) (cells × features/genes)
_Matrix density_: 0.03319170246582202
* Meaning only ~3.3% of the expression matrix entries are non-zeros


* _obs_ : cell meta-data (nCount_RNA, nFeature_RNA, Variant_Group etc.)
* _obsm_ : cell-level embeddings
    * X_pca -> PCA-coordinates
    * X_umap -> UMAP coordinates
    * more...

* _adata.X_ : Expression matrix (cell x feature) - "sparse"

In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

In [ ]:
# Load file
FILE_PATH = "data/Variant_Vax_obj.h5ad"
adata = sc.read_h5ad(FILE_PATH)

In [ ]:
print(adata)

AnnData object with n_obs × n_vars = 48730 × 29961
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'Variant_Group', 'Participant', 'SARSCoV2_PCR_Status', 'Vaccination_Status', 'WHO_Score_at_Peak', 'SingleCell_SARSCoV2_RNA_Status', 'Coarse_Annotation', 'Detailed_Annotation', 'Variant_Vax_Group'
    obsm: 'X_harmony', 'X_pca', 'X_umap'


In [18]:
type(adata.X)
adata.X[:5, :5].A if hasattr(adata.X, "A") else adata.X[:5, :5]
adata.layers.keys()


KeysView(Layers with keys: )

In [11]:
print(adata.obs)

                                       nCount_RNA  nFeature_RNA  percent.mt  \
Ancestral_A_COVID19_01_CAACGCGGGTTG-1         824           704    0.000000   
Ancestral_A_COVID19_01_CAGACCTTCTGT-1        1573          1144    0.000000   
Ancestral_A_COVID19_01_CTAGCCAGACCA-1         257           218    0.000000   
Ancestral_A_COVID19_01_GGGTACTTGGGA-1        1812          1249    8.112583   
Ancestral_A_COVID19_01_GAAAGCAGTGTG-1         388           290    0.000000   
...                                           ...           ...         ...   
Omicron_O_COVID19_07__ACCCCAAGTGCG-27        2714          1306    0.000000   
Omicron_O_COVID19_07__TAATTTATCGTG-27         698           498    0.143266   
Omicron_O_COVID19_07__GTTACCAGCTCG-27        1166           722    0.000000   
Omicron_O_COVID19_07__TGGCGATTTCAT-27         640           470    0.000000   
Omicron_O_COVID19_07__AATACCTACTGT-27         912           639    0.000000   

                                      Variant_Group

In [ ]:
# Basic dimensions
print("Shape (cells × features/genes):", adata.shape)

# Matrix info 
X = adata.X
print("\nMatrix type:", type(X))
print("Matrix dtype:", X.dtype)
try:
    print("Matrix density:", X.nnz / (X.shape[0] * X.shape[1]))
except AttributeError:
    print("Matrix density: dense matrix")

# obs metadata
print("\nobs columns:", list(adata.obs.columns))
print("obs head:")
display(adata.obs.head())

# --- var metadata ---
print("\nvar columns:", list(adata.var.columns))
print("var head:")
display(adata.var.head())

# --- QC metrics if present ---
qc_fields = [c for c in adata.obs.columns if "n_" in c or "pct" in c or "mt" in c]
print("\nQC-like fields:", qc_fields)

Shape (cells × features/genes): (48730, 29961)

Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Matrix dtype: int64
Matrix density: 0.03319170246582202

obs columns: ['nCount_RNA', 'nFeature_RNA', 'percent.mt', 'Variant_Group', 'Participant', 'SARSCoV2_PCR_Status', 'Vaccination_Status', 'WHO_Score_at_Peak', 'SingleCell_SARSCoV2_RNA_Status', 'Coarse_Annotation', 'Detailed_Annotation', 'Variant_Vax_Group']
obs head:


,nCount_RNA,nFeature_RNA,percent.mt,Variant_Group,Participant,SARSCoV2_PCR_Status,Vaccination_Status,WHO_Score_at_Peak,SingleCell_SARSCoV2_RNA_Status,Coarse_Annotation,Detailed_Annotation,Variant_Vax_Group
Ancestral_A_COVID19_01_CAACGCGGGTTG-1,824,704,0.000000,Ancestral,A_COVID19_01,positive,unvaccinated,8,SARSCoV2 RNA-,Ciliated,Ciliated.SERPINB3.VMO1.AQP5,Ancestral
Ancestral_A_COVID19_01_CAGACCTTCTGT-1,1573,1144,0.000000,Ancestral,A_COVID19_01,positive,unvaccinated,8,SARSCoV2 RNA-,Deuterosomal,Deuterosomal,Ancestral
Ancestral_A_COVID19_01_CTAGCCAGACCA-1,257,218,0.000000,Ancestral,A_COVID19_01,positive,unvaccinated,8,SARSCoV2 RNA-,Secretory,Secretory.SARSCoV2,Ancestral
Ancestral_A_COVID19_01_GGGTACTTGGGA-1,1812,1249,8.112583,Ancestral,A_COVID19_01,positive,unvaccinated,8,SARSCoV2 RNA-,Ciliated,Ciliated.JUN.FOS.DNAJB1,Ancestral
Ancestral_A_COVID19_01_GAAAGCAGTGTG-1,388,290,0.000000,Ancestral,A_COVID19_01,positive,unvaccinated,8,SARSCoV2 RNA-,Secretory,Secretory.NEAT1.ANKRD36C.PNISR,Ancestral



var columns: []
var head:


""
gene
A1BG
A1BG-AS1
A1CF
A2M
A2M-AS1



.uns keys: []

QC-like fields: ['percent.mt', 'Vaccination_Status']


In [20]:
import pandas as pd

# Extract patient → WHO score mapping
df = adata.obs[["Participant", "WHO_Score_at_Peak"]].copy()

# Drop cells with missing WHO score
df = df.dropna(subset=["WHO_Score_at_Peak"])

# Collapse to patient-level (each patient should have one WHO score)
patient_scores = df.groupby("Participant")["WHO_Score_at_Peak"].first()

# Summary stats
print("Min WHO score:", patient_scores.min())
print("Max WHO score:", patient_scores.max())

# Distribution
print("\nDistribution (counts):")
print(patient_scores.value_counts().sort_index())

print("\nDistribution (percent):")
print((patient_scores.value_counts(normalize=True).sort_index() * 100).round(2))


Min WHO score: 0
Max WHO score: 8

Distribution (counts):
WHO_Score_at_Peak
0    27
1     4
2     1
3     9
4    17
5    12
6     3
7     8
8    31
Name: count, dtype: int64

Distribution (percent):
WHO_Score_at_Peak
0    24.11
1     3.57
2     0.89
3     8.04
4    15.18
5    10.71
6     2.68
7     7.14
8    27.68
Name: proportion, dtype: float64
